# 02 — Data Acquisition & Initial Data Quality / Bài tập 2

Mục tiêu: Pandas I/O, NumPy normalization, raw → interim → processed,
missing/duplicate/inconsistency checks và benchmark định dạng.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dap_project.data.organize import (
    normalize_dataframe, build_processed, quality_summary,
    save_multiformat, benchmark_io
)

In [ ]:
raw = pd.read_csv(ROOT / "data/raw/cafeland_pilot_verified.csv")
interim = normalize_dataframe(raw, ROOT / "docs/hcm_pre2025_scope_map.json")
processed = build_processed(interim)

print("raw:", raw.shape)
print("interim:", interim.shape)
print("processed:", processed.shape)

In [ ]:
# Missing rates
missing_rate = interim.isna().mean().sort_values(ascending=False)
missing_rate

In [ ]:
# Duplicate checks
{
    "duplicate_rows": int(interim.duplicated().sum()),
    "duplicate_listing_id": int(interim["listing_id"].duplicated().sum()),
    "invalid_price": int((~interim["price_valid"]).sum()),
    "invalid_area": int((~interim["area_valid"]).sum()),
    "scope_unresolved": int((interim["scope_status"] == "unresolved").sum()),
    "scope_conflict": int((interim["scope_status"] == "conflict").sum()),
    "area_definition_risk": int(interim["area_definition_risk"].sum()),
}

In [ ]:
# NumPy vectorization example: price per m2 ONLY for EDA/QA, not model input.
price = processed["price_vnd"].to_numpy(dtype=float)
area = processed["area_m2"].to_numpy(dtype=float)
processed["qa_price_per_m2"] = np.divide(
    price, area, out=np.full_like(price, np.nan), where=area > 0
)
processed[["price_vnd", "area_m2", "qa_price_per_m2"]].head()

In [ ]:
# Multi-format save + benchmark
paths = save_multiformat(
    processed, ROOT / "data/processed/cafeland_pilot_cleaned_benchmark"
)
benchmark_io(paths)

## Gate production

Pilot không được dùng để tuyên bố đạt ≥5.000 dòng.

Sau khi chạy production:

```bash
python scripts/run_all.py --stage acquisition --max-records 6500
python scripts/run_all.py --stage quality
```

Chỉ khi processed dataset thực tế >= 5.000 dòng và >= 8 biến mới đánh dấu Bài tập 2 qua gate.